#Stock Indicators Python

In [1]:
import pandas as pd
import requests
import numpy as np
from lightweight_charts import Chart
from stock_indicators import indicators, Quote
from datetime import datetime, timedelta
import asyncio
import nest_asyncio

nest_asyncio.apply()

In [2]:
import yfinance as yf
df = yf.download('TQQQ', start='2010-01-01', multi_level_index=False)
df.reset_index(inplace=True)
df.to_csv('TQQQ_data.csv', index=False)
df = pd.read_csv('TQQQ_data.csv')
rawdf = df.copy()
df['Date'] = pd.to_datetime(df['Date'])
df.head()

C:\Users\jwang\AppData\Local\Temp\ipykernel_28156\15803499.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('TQQQ', start='2010-01-01', multi_level_index=False)
[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2010-02-11,0.413438,0.415679,0.387652,0.388896,3456000
1,2010-02-12,0.415131,0.418715,0.399848,0.402187,8601600
2,2010-02-16,0.431211,0.432207,0.418217,0.424888,9619200
3,2010-02-17,0.438528,0.438628,0.430414,0.436986,19180800
4,2010-02-18,0.446842,0.449480,0.435442,0.438080,38860800


In [3]:
quotes = [
    Quote(d, o, h, l, c, v)
    for d, o, h, l, c, v in zip(
        df['Date'],
        df['Open'],
        df['High'],
        df['Low'],
        df['Close'],
        df['Volume']
    )
]


In [4]:
# ATR Trailing Stop
df['atr_stop'] = [r.atr_stop for r in indicators.get_atr_stop(quotes)]
df['atr_stop'] = df['atr_stop'].astype(float) 
df.to_json()
df['ATRStop_signal'] = np.where(df['Close'] > df['atr_stop'], 'long', 'short')

# Average Directional Index (ADX)
df['adx'] = [r.adx for r in indicators.get_adx(quotes, 14)]
df['adx_pdi'] = [r.pdi for r in indicators.get_adx(quotes, 14)]
df['adx_mdi'] = [r.mdi for r in indicators.get_adx(quotes, 14)]
df['ADX_signal'] = np.where(df['adx_pdi'] > df['adx_mdi'], 'long', 'short')

# Moving Average Convergence / Divergence (MACD)
df['macd'] = [r.macd for r in indicators.get_macd(quotes, 12, 26, 9)]
df['signal'] = [r.signal for r in indicators.get_macd(quotes, 12, 26, 9)]
df['histogram'] = [r.histogram for r in indicators.get_macd(quotes, 12, 26, 9)]
df['MACD_signal'] = np.where(df['macd'] > df['signal'], 'long', 'short')

# Williams Alligator
df['jaw'] = [r.jaw for r in indicators.get_alligator(quotes, 13, 8, 8, 5, 5, 3)]
df['teeth'] = [r.teeth for r in indicators.get_alligator(quotes, 13, 8, 8, 5, 5, 3)]
df['lips'] = [r.lips for r in indicators.get_alligator(quotes, 13, 8, 8, 5, 5, 3)]

# Vortex Indicator (VI)
df['pvi'] = [r.pvi for r in indicators.get_vortex(quotes, 14)]
df['nvi'] = [r.nvi for r in indicators.get_vortex(quotes, 14)]
df['VI_signal'] = np.where(df['pvi'] > df['nvi'], 'long', 'short')

# Calculate EMA
df['EMA 12'] = [r.ema for r in indicators.get_ema(quotes, 12)]
df['EMA 20'] = [r.ema for r in indicators.get_ema(quotes, 20)]
df['EMA 25'] = [r.ema for r in indicators.get_ema(quotes, 25)]
df['EMA_signal'] = np.where(df['EMA 12'] > df['EMA 25'], 'long', 'short')

# Stochastic Momentum Index (SMI)
df['smi'] = [r.smi for r in indicators.get_smi(quotes, 14, 3, 3)]
df['smi_signal'] = [r.signal for r in indicators.get_smi(quotes, 14, 3, 3)]
df['SMI_signal'] = np.where(df['smi'] > df['smi_signal'], 'long', 'short')

# Calculate Chandelier exit
from stock_indicators import ChandelierType 
df['chandelier_long_exit'] = [r.chandelier_exit for r in indicators.get_chandelier(quotes, 22, 3, ChandelierType.LONG)]
df['chandelier_short_exit'] = [r.chandelier_exit for r in indicators.get_chandelier(quotes, 22, 3, ChandelierType.SHORT)]
df['chandelier_exit_signal'] = np.where(df['chandelier_long_exit'] > df['chandelier_short_exit'], 'long', 'short')

# Calculate Bollinger Bands
df['BB_upper_band'] = [r.upper_band for r in indicators.get_bollinger_bands(quotes, 20, 2)]
df['BB_middle_band'] = [r.sma for r in indicators.get_bollinger_bands(quotes, 20, 2)]
df['BB_lower_band'] = [r.lower_band for r in indicators.get_bollinger_bands(quotes, 20, 2)]

# Arnaud Legoux Moving Average (ALMA)
df['ALMA_21'] = [r.alma for r in indicators.get_alma(quotes, 21, 0.85, 6)]
df['ALMA_50'] = [r.alma for r in indicators.get_alma(quotes, 50, 0.85, 6)]
df['alma_signal'] = np.where(df['ALMA_21'] > df['ALMA_50'], 'long', 'short')


# Ehlers Fisher Transform
df['fisher'] = [r.fisher for r in indicators.get_fisher_transform(quotes, 10)]
df['fisher_trigger'] = [r.trigger for r in indicators.get_fisher_transform(quotes, 10)]
df['Fisher_signal'] = np.where(df['fisher'] > df['fisher_trigger'], 'long', 'short')

# Choppiness Index
df['chopp'] = [r.chop for r in indicators.get_chop(quotes, 14)]

# Price Momentum Oscillator (PMO)
df['pmo'] = [r.pmo for r in indicators.get_pmo(quotes, 35, 20, 10)]
df['pmo_signal'] = [r.signal for r in indicators.get_pmo(quotes, 35, 20, 10)]
df['PMO_signal'] = np.where(df['pmo'] > df['pmo_signal'], 'long', 'short')

# Calculate RSI
df['rsi'] = [r.rsi for r in indicators.get_rsi(quotes, 14)]
df['rsima6'] = df['rsi'].rolling(6).mean()
df['rsima14'] = df['rsi'].rolling(14).mean()
df['rsi_signal'] = np.where(df['rsi'] > df['rsima14'], 'long', 'short')

#True Strength Index (TSI)
df['tsi'] = [r.tsi for r in indicators.get_tsi(quotes, 25, 13)]
df['tsi_signal'] = [r.signal for r in indicators.get_tsi(quotes, 25, 13)]
df['TSI_signal'] = np.where(df['tsi'] > df['tsi_signal'], 'long', 'short')

# Calculate Supertrend
df['supertrend'] = [r.super_trend for r in indicators.get_super_trend(quotes, 14, 3)]
df['supertrend_signal'] = np.where(df['supertrend'] > df['Close'], 'short', 'long')

# Calculate Mcginley dynamic
df['dynamic20'] = [r.dynamic for r in indicators.get_dynamic(quotes, 20)]
df['McGinley_signal'] = np.where(df['Close'] > df['dynamic20'], 'long', 'short')

In [5]:
df['atr_stop'] = df['atr_stop'].astype(str) 
df['supertrend'] = df['supertrend'].astype(str)

# Now you can serialize your DataFrame
df.to_json() 


df = df.dropna().reset_index(drop=True)
df.head()


,Date,Close,High,Low,Open,Volume,atr_stop,ATRStop_signal,adx,adx_pdi,...,rsima6,rsima14,rsi_signal,tsi,tsi_signal,TSI_signal,supertrend,supertrend_signal,dynamic20,McGinley_signal
0,2010-05-13,0.529928,0.561241,0.526593,0.549691,52780800,0.563913614969314,short,37.484090,14.718129,...,40.795089,49.131177,short,-8.774725,-1.839088,short,0.566729344505653,short,0.537474,short
1,2010-05-14,0.497321,0.517632,0.480395,0.516935,77088000,0.563913614969314,short,38.480185,13.508893,...,41.964524,46.683759,short,-10.715387,-4.058163,short,0.566729344505653,short,0.532909,short
2,2010-05-17,0.504042,0.509617,0.471932,0.502150,68428800,0.563913614969314,short,39.496917,12.656890,...,44.065524,45.758477,short,-11.735952,-5.977610,short,0.566729344505653,short,0.529903,short
3,2010-05-18,0.484776,0.519225,0.475915,0.516337,95308800,0.563913614969314,short,40.099236,13.346703,...,43.279711,44.612145,short,-13.439739,-7.843142,short,0.566729344505653,short,0.524535,short
4,2010-05-19,0.471982,0.487863,0.455156,0.476413,118886400,0.563913614969314,short,40.911279,12.604461,...,42.220646,42.729454,short,-15.347664,-9.719273,short,0.566729344505653,short,0.517854,short


In [6]:
# ATR Trailing Stop
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    atr_stop_line = chart.create_line('atr_stop', color="#d62728", width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])
   
   # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atrStop_signal = df.iloc[i]['ATRStop_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if atrStop_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif atrStop_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [7]:
# ATR Trailing Stop and SMI
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop and SMI", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    atr_stop_line = chart.create_line('atr_stop', color="#d62728", width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])
    
    # Create line series for SMIs
    smi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    smi_line = smi_chart.create_line('smi', color="#2ee30f", width=1, price_line=False, price_label=False)
    smi_line.set(df[['Date', 'smi']])

    signal_line = smi_chart.create_line('smi_signal', color="#f36021", width=1, price_line=False, price_label=False)
    signal_line.set(df[['Date', 'smi_signal']])
   
   # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atrStop_signal = df.iloc[i]['ATRStop_signal']
        SMI_signal = df.iloc[i]['SMI_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if atrStop_signal == 'long' and SMI_signal == "long" and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif atrStop_signal == 'short' and SMI_signal == 'short' and  sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [8]:
# Average Directional Index (ADX)
if __name__ == '__main__':
    
    chart = Chart(title="ADX Crossover", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    adx_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    adx_line = adx_chart.create_line('adx', color="#e3b90f", width=1, price_line=False, price_label=False)
    adx_line.set(df[['Date', 'adx']])
    adx_pdi_line = adx_chart.create_line('adx_pdi', color="#75f321", width=1, price_line=False, price_label=False)
    adx_pdi_line.set(df[['Date', 'adx_pdi']])
    adx_mdi_line = adx_chart.create_line('adx_mdi', color="#f40707", width=1, price_line=False, price_label=False)
    adx_mdi_line.set(df[['Date', 'adx_mdi']])   

    


    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        adx_signal = df.iloc[i]['ADX_signal']
        adx_value = df.iloc[i]['adx']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if adx_signal == 'long' and adx_value > 25  and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif adx_signal == 'short' and adx_value > 25 and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [9]:
# Vortex Indicator (VI)
if __name__ == '__main__':
    
    chart = Chart(title="VI Crossover", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    vi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    vi_pvi_line = vi_chart.create_line('pvi', color="#75f321", width=1, price_line=False, price_label=False)
    vi_pvi_line.set(df[['Date', 'pvi']])
    vi_nvi_line = vi_chart.create_line('nvi', color="#f40707", width=1, price_line=False, price_label=False)
    vi_nvi_line.set(df[['Date', 'nvi']])
    

    
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        vi_signal = df.iloc[i]['VI_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if vi_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif vi_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [10]:
# Moving Average Convergence / Divergence (MACD)
if __name__ == '__main__':
    
    chart = Chart(title="MACD Crossover", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    macd_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    macd_line = macd_chart.create_line('macd', color="#2ee30f", width=1, price_line=False, price_label=False)
    macd_line.set(df[['Date', 'macd']])

    signal_line = macd_chart.create_line('signal', color="#f36021", width=1, price_line=False, price_label=False)
    signal_line.set(df[['Date', 'signal']])

    histogram_bar = macd_chart.create_histogram('histogram', color="#1341e4", price_line=False, price_label=False)
    histogram_bar.set(df[['Date', 'histogram']])


    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        MACD_signal = df.iloc[i]['MACD_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if MACD_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif MACD_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [11]:
# Williams Alligator
if __name__ == '__main__':
    
    chart = Chart(title="Williams Alligator", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for Williams Alligator
    jaw_line = chart.create_line('jaw', color="#1f77b4", width=1, price_line=False, price_label=False)
    jaw_line.set(df[['Date', 'jaw']])
    teeth_line = chart.create_line('teeth', color="#ff7f0e", width=1, price_line=False, price_label=False)
    teeth_line.set(df[['Date', 'teeth']])
    lips_line = chart.create_line('lips', color="#2ca02c", width=1, price_line=False, price_label=False)
    lips_line.set(df[['Date', 'lips']])

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [12]:
# Bollinger Bands
if __name__ == '__main__':

    chart = Chart(title="Bollinger Bands", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    upper_band_line = chart.create_line('BB_upper_band', color="#ff0000", width=1, price_line=False, price_label=False)
    upper_band_line.set(df[['Date', 'BB_upper_band']])
    middle_band_line = chart.create_line('BB_middle_band', color="#f3bd0b", width=1, price_line=False, price_label=False)
    middle_band_line.set(df[['Date', 'BB_middle_band']])
    lower_band_line = chart.create_line('BB_lower_band', color="#00ff00", width=1, price_line=False, price_label=False)
    lower_band_line.set(df[['Date', 'BB_lower_band']])

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [13]:
# Stochastic Momentum Index (SMI)
if __name__ == '__main__':
    
    chart = Chart(title="Stochastic Momentum Index (SMI)", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for SMIs
    smi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    smi_line = smi_chart.create_line('smi', color="#2ee30f", width=1, price_line=False, price_label=False)
    smi_line.set(df[['Date', 'smi']])

    signal_line = smi_chart.create_line('smi_signal', color="#f36021", width=1, price_line=False, price_label=False)
    signal_line.set(df[['Date', 'smi_signal']])

    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        SMI_signal = df.iloc[i]['SMI_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if SMI_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif SMI_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [14]:
# Chandelier Exit

if __name__ == '__main__':
    
    chart = Chart(title="chandelier long_exit and short_exit Crossover", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Add Chandelier Exit lines
    chandelier_long_line = chart.create_line('Chandelier_Long_Exit', color='#8e44ad', width=1, price_line=False, price_label=False)
    chandelier_long_line.set(df[['Date', 'chandelier_long_exit']])
    # chandelier_short_line = chart.create_line('Chandelier_Short_Exit', color='#e74c3c', width=1, price_line=False, price_label=False)
    # chandelier_short_line.set(df[['Date', 'chandelier_short_exit']])

    
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        chan_signal = df.iloc[i]['chandelier_exit_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if chan_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif chan_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [15]:
# Now, let's create the chart with lightweight-charts and add the EMAs and buy/sell markers.

# Assuming `df` is already a pandas DataFrame with 'Date', 'Open', 'High', 'Low', 'Close', 'EMA 12', and 'EMA 25' columns.
# It's good practice to convert the 'Date' column to the correct datetime format.
# df['Date'] = pd.to_datetime(df['Date'])

if __name__ == '__main__':
    
    chart = Chart(title="EMA12_EMA25 Crossover", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    ema12_line = chart.create_line('EMA 12', color='#ffeb3b', width=1, price_line=False, price_label=False)
    ema12_line.set(df[['Date', 'EMA 12']])

    ema25_line = chart.create_line('EMA 25', color='#26c6da', width=1, price_line=False, price_label=False)
    ema25_line.set(df[['Date', 'EMA 25']])
              
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        ema_signal = df.iloc[i]['EMA_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if ema_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0            
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif ema_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [16]:
# Arnaud Legoux Moving Average (ALMA)

if __name__ == '__main__':
    
    chart = Chart(title="Arnaud Legoux Moving Average (ALMA)", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    alma21_line = chart.create_line('ALMA_21', color='#ffeb3b', width=1, price_line=False, price_label=False)
    alma21_line.set(df[['Date', 'ALMA_21']])

    alma50_line = chart.create_line('ALMA_50', color='#26c6da', width=1, price_line=False, price_label=False)
    alma50_line.set(df[['Date', 'ALMA_50']])
              
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        alma_diff = df.iloc[i]['alma_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if alma_diff == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif alma_diff == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [17]:
# Ehlers Fisher Transform

if __name__ == '__main__':
    
    chart = Chart(title="Ehlers Fisher Transform", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for fisher transform
    fisher_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    fisher_line = fisher_chart.create_line('fisher', color='#ffeb3b', width=1, price_line=False, price_label=False)
    fisher_line.set(df[['Date', 'fisher']])

    fisher_trigger_line = fisher_chart.create_line('fisher_trigger', color='#26c6da', width=1, price_line=False, price_label=False)
    fisher_trigger_line.set(df[['Date', 'fisher_trigger']])
              
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        fisher_signal = df.iloc[i]['Fisher_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if fisher_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif fisher_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [18]:
# Choppiness Index

if __name__ == '__main__':

    chart = Chart(title="Choppiness Index", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for choppiness index
    chop_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    chop_line = chop_chart.create_line('chopp', color='#ffeb3b', width=1, price_line=False, price_label=False)
    chop_line.set(df[['Date', 'chopp']])
    # chop60_line = chop_chart.horizontal_line(price=60, color='#26c6da', width=1, style='dashed', text='60')
    # chop40_line = chop_chart.horizontal_line(price=40, color='#ff5733', width=1, style='dashed', text='40')

chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [19]:
# Price Momentum Oscillator (PMO)

if __name__ == '__main__':

    chart = Chart(title="Price Momentum Oscillator (PMO)", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for PMO
    pmo_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    pmo_line = pmo_chart.create_line('pmo', color='#ffeb3b', width=1, price_line=False, price_label=False)
    pmo_line.set(df[['Date', 'pmo']])

    pmo_signal_line = pmo_chart.create_line('pmo_signal', color='#26c6da', width=1, price_line=False, price_label=False)
    pmo_signal_line.set(df[['Date', 'pmo_signal']])
              
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        pmo_signal = df.iloc[i]['PMO_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if pmo_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif pmo_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)
    
chart.show(block = True)


c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [20]:
# True Strength Index (TSI)
if __name__ == '__main__':

    chart = Chart(title="True Strength Index (TSI)", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for TSI
    tsi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    tsi_line = tsi_chart.create_line('tsi', color='#ffeb3b', width=1, price_line=False, price_label=False)
    tsi_line.set(df[['Date', 'tsi']])

    tsi_signal_line = tsi_chart.create_line('tsi_signal', color='#26c6da', width=1, price_line=False, price_label=False)
    tsi_signal_line.set(df[['Date', 'tsi_signal']])
              
    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        tsi_signal = df.iloc[i]['TSI_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if tsi_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif tsi_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [21]:
df['supertrend'] = df['supertrend'].astype(str)

# Now you can serialize your DataFrame
df.to_json() 

'{"Date":{"0":1273708800000,"1":1273795200000,"2":1274054400000,"3":1274140800000,"4":1274227200000,"5":1274313600000,"6":1274400000000,"7":1274659200000,"8":1274745600000,"9":1274832000000,"10":1274918400000,"11":1275004800000,"12":1275350400000,"13":1275436800000,"14":1275523200000,"15":1275609600000,"16":1275868800000,"17":1275955200000,"18":1276041600000,"19":1276128000000,"20":1276214400000,"21":1276473600000,"22":1276560000000,"23":1276646400000,"24":1276732800000,"25":1276819200000,"26":1277078400000,"27":1277164800000,"28":1277251200000,"29":1277337600000,"30":1277424000000,"31":1277683200000,"32":1277769600000,"33":1277856000000,"34":1277942400000,"35":1278028800000,"36":1278374400000,"37":1278460800000,"38":1278547200000,"39":1278633600000,"40":1278892800000,"41":1278979200000,"42":1279065600000,"43":1279152000000,"44":1279238400000,"45":1279497600000,"46":1279584000000,"47":1279670400000,"48":1279756800000,"49":1279843200000,"50":1280102400000,"51":1280188800000,"52":1280275

In [22]:
# Supertrend
if __name__ == '__main__':

    chart = Chart(title="Supertrend", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for Supertrend

    supertrend_line = chart.create_line('supertrend', color='#ffeb3b', width=1, price_line=False, price_label=False)
    supertrend_line.set(df[['Date', 'supertrend']])

    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        supertrend_signal = df.iloc[i]['supertrend_signal']
        
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if supertrend_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif supertrend_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [23]:
# Supertrend and McGinley Dynamic
if __name__ == '__main__':

    chart = Chart(title="Supertrend and McGinley Dynamic", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for Supertrend

    supertrend_line = chart.create_line('supertrend', color='#ffeb3b', width=1, price_line=False, price_label=False)
    supertrend_line.set(df[['Date', 'supertrend']])

    dynamic20_line = chart.create_line('dynamic20', color='#26c6da', width=1, price_line=False, price_label=False)
    dynamic20_line.set(df[['Date', 'dynamic20']])

    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        supertrend_signal = df.iloc[i]['supertrend_signal']
        dynamic20_signal = df.iloc[i]['McGinley_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if supertrend_signal == 'long' and dynamic20_signal == 'long'and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif supertrend_signal == 'short' and dynamic20_signal == 'short' and sell_signal == 0 :
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [24]:
# Supertrend and McGinley Dynamic and RSI
if __name__ == '__main__':

    chart = Chart(title="Supertrend and McGinley Dynamic and RSI", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for Supertrend

    supertrend_line = chart.create_line('supertrend', color='#ffeb3b', width=1, price_line=False, price_label=False)
    supertrend_line.set(df[['Date', 'supertrend']])

    dynamic20_line = chart.create_line('dynamic20', color='#26c6da', width=1, price_line=False, price_label=False)
    dynamic20_line.set(df[['Date', 'dynamic20']])

    ris_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    rsi_line = ris_chart.create_line('rsi', color='#ff5733', width=1, price_line=False, price_label=False)
    rsi_line.set(df[['Date', 'rsi']])

    rsi_ma14_line = ris_chart.create_line('rsima14', color='#33ff57', width=1, price_line=False, price_label=False)
    rsi_ma14_line.set(df[['Date', 'rsima14']])

    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        supertrend_signal = df.iloc[i]['supertrend_signal']
        dynamic20_signal = df.iloc[i]['McGinley_signal']
        rsi_signal = df.iloc[i]['rsi_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if supertrend_signal == 'long' and dynamic20_signal == 'long' and rsi_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif supertrend_signal == 'short' and dynamic20_signal == 'short' and rsi_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal =0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [25]:
# Supertrend and RSI
if __name__ == '__main__':

    chart = Chart(title="Supertrend and RSI", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)
    # chart.layout(background_color="white")

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for Supertrend

    supertrend_line = chart.create_line('supertrend', color='#ffeb3b', width=1, price_line=False, price_label=False)
    supertrend_line.set(df[['Date', 'supertrend']])

    
    ris_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    rsi_line = ris_chart.create_line('rsi', color='#ff5733', width=1, price_line=False, price_label=False)
    rsi_line.set(df[['Date', 'rsi']])

    rsi_ma14_line = ris_chart.create_line('rsima14', color='#33ff57', width=1, price_line=False, price_label=False)
    rsi_ma14_line.set(df[['Date', 'rsima14']])

    # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        supertrend_signal = df.iloc[i]['supertrend_signal']
        rsi_signal = df.iloc[i]['rsi_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if supertrend_signal == 'long' and rsi_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif supertrend_signal == 'short' and rsi_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block=True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [26]:
# ATR Trailing Stop and SMI
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop and SMI", maximize=True)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for EMAs
    atr_stop_line = chart.create_line('atr_stop', color="#d62728", width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])
    
    # Create line series for SMIs
    smi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    smi_line = smi_chart.create_line('smi', color="#2ee30f", width=1, price_line=False, price_label=False)
    smi_line.set(df[['Date', 'smi']])

    signal_line = smi_chart.create_line('smi_signal', color="#f36021", width=1, price_line=False, price_label=False)
    signal_line.set(df[['Date', 'smi_signal']])
   
   # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atrStop_signal = df.iloc[i]['ATRStop_signal']
        SMI_signal = df.iloc[i]['SMI_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if atrStop_signal == 'long' and SMI_signal == "long" and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif atrStop_signal == 'short' and SMI_signal == 'short' and  sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [27]:
# ATR Trailing Stop and SMI and Ehlers Fisher Transform
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop and SMI and Fisher Transform", maximize=True, inner_height=0.6)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for ATR_trailing_stop
    atr_stop_line = chart.create_line('atr_stop', color="#d62728", width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])
    
    # Create line series for SMIs
    smi_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    smi_line = smi_chart.create_line('smi', color="#2ee30f", width=1, price_line=False, price_label=False)
    smi_line.set(df[['Date', 'smi']])
    signal_line = smi_chart.create_line('smi_signal', color="#f36021", width=1, price_line=False, price_label=False)
    signal_line.set(df[['Date', 'smi_signal']])
    
    # Create line series for fisher transform
    fisher_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    fisher_line = fisher_chart.create_line('fisher', color='#ffeb3b', width=1, price_line=False, price_label=False)
    fisher_line.set(df[['Date', 'fisher']])
    fisher_trigger_line = fisher_chart.create_line('fisher_trigger', color='#26c6da', width=1, price_line=False, price_label=False)
    fisher_trigger_line.set(df[['Date', 'fisher_trigger']])
   
   # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atrStop_signal = df.iloc[i]['ATRStop_signal']
        SMI_signal = df.iloc[i]['SMI_signal']
        fisher_signal = df.iloc[i]['Fisher_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if atrStop_signal == 'long' and SMI_signal == "long" and fisher_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif atrStop_signal == 'short' and SMI_signal == 'short' and fisher_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')


In [28]:
# ATR Trailing Stop and Ehlers Fisher Transform
if __name__ == '__main__':

    chart = Chart(title="ATR Trailing Stop and Fisher Transform", maximize=True, inner_height=0.8)
    chart.legend(visible=True, color_based_on_candle=True)

    # Set the main candlestick data for the chart.
    # The 'lightweight-charts' library expects a DataFrame with columns like 'Date', 'Open', 'High', 'Low', 'Close'.
    chart.set(df)

    # Create line series for ATR_trailing_stop
    atr_stop_line = chart.create_line('atr_stop', color="#d62728", width=1, price_line=False, price_label=False)
    atr_stop_line.set(df[['Date', 'atr_stop']])
    
    # Create line series for fisher transform
    fisher_chart = chart.create_subchart(position='left', width=1.0, height=0.2, sync=True)
    fisher_line = fisher_chart.create_line('fisher', color='#ffeb3b', width=1, price_line=False, price_label=False)
    fisher_line.set(df[['Date', 'fisher']])
    fisher_trigger_line = fisher_chart.create_line('fisher_trigger', color='#26c6da', width=1, price_line=False, price_label=False)
    fisher_trigger_line.set(df[['Date', 'fisher_trigger']])
   
   # Initialize a list to hold the markers
    markers = []
    buy_signal = 0
    sell_signal = 0

    # Iterate through the DataFrame to find crossover points
    for i in range(1, len(df)):

        atrStop_signal = df.iloc[i]['ATRStop_signal']
        SMI_signal = df.iloc[i]['SMI_signal']
        fisher_signal = df.iloc[i]['Fisher_signal']
        current_time = df.iloc[i]['Date']

        # Check for buy signal (EMA 12 crosses above EMA 25)
        if atrStop_signal == 'long' and fisher_signal == 'long' and buy_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'below',
                'shape': 'arrow_up',
                'color': '#33de3d',
                'text': 'Buy'
            })
            buy_signal = 1
            sell_signal = 0
        
        # Check for sell signal (EMA 12 crosses below EMA 25)
        elif atrStop_signal == 'short' and fisher_signal == 'short' and sell_signal == 0:
            markers.append({
                'time': current_time,
                'position': 'above',
                'shape': 'arrow_down',
                'color': '#f485fb',
                'text': 'Sell'
            })
            sell_signal = 1
            buy_signal = 0

    # Add all markers at once. It's more efficient than adding them individually in a loop.
    if markers:
        chart.marker_list(markers)

chart.show(block = True)

c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
c:\Users\jwang\AppData\Local\Programs\Python\Python313\Lib\site-packages\lightweight_charts\util.py:41: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  d = data.to_dict(orient='records')
